# Day 046 — Exercise 2: Resampling

**What you'll build:** `resample_series(series, freq, agg='sum') -> pd.Series` — aggregate a datetime-indexed Series at a new time frequency (daily, weekly, monthly). And `multi_freq_summary(series) -> dict` — return a dict of the series summarised at daily and weekly frequencies.

**Why it matters:** Raw data often arrives daily but insights live at the weekly or monthly level. Resampling is how you move between them — it's the time-series equivalent of `groupby`.

## Provided: Setup + parse_time_series

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

def make_sample_ts(n_days: int = 90, seed: int = 42) -> pd.DataFrame:
    """Return a reproducible daily time-series DataFrame for exercises."""
    rng   = np.random.default_rng(seed)
    dates = pd.date_range('2024-01-01', periods=n_days, freq='D')
    vals  = 1000.0 + (rng.standard_normal(n_days).cumsum() * 50)
    return pd.DataFrame({'date': dates.strftime('%Y-%m-%d'),
                         'value': vals.round(2)})


import pandas as pd
import warnings
warnings.filterwarnings('ignore')

def parse_time_series(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    df = df.dropna(subset=[date_col])
    df = df.set_index(date_col).sort_index()
    return df

def date_features(df: pd.DataFrame) -> pd.DataFrame:
    idx = df.index
    return pd.DataFrame({
        'year':        idx.year,
        'month':       idx.month,
        'day':         idx.day,
        'day_of_week': idx.dayofweek,
        'quarter':     idx.quarter,
    }, index=idx)

## Your Implementation

In [ ]:
def resample_series(series: pd.Series, freq: str,
                    agg: str = 'sum') -> pd.Series:
    """
    Resample a datetime-indexed Series to a new frequency.

    Args:
        series: pd.Series with DatetimeIndex
        freq:   resample rule — 'D' daily, 'W' weekly, 'ME' month-end
        agg:    aggregation — 'sum', 'mean', 'min', 'max', 'count'
    Returns:
        Resampled pd.Series
    """
    # TODO: return series.resample(freq).agg(agg)
    pass


def multi_freq_summary(series: pd.Series) -> dict:
    """
    Return dict with keys 'daily' and 'weekly', each a resampled sum Series.
    """
    # TODO: return {
    #     'daily':  resample_series(series, 'D'),
    #     'weekly': resample_series(series, 'W'),
    # }
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    df_raw = make_sample_ts(28)
    df_ts  = parse_time_series(df_raw, 'date')
    series = df_ts['value']

    # Check 1: resample_series defined, returns pd.Series
    try:
        assert 'resample_series' in globals()
        daily = resample_series(series, 'D')
        assert isinstance(daily, pd.Series), \
            f'expected Series, got {type(daily).__name__}'
        passed += 1; print('\u2705 Check 1: resample_series returns pd.Series')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: daily resample — same length as input (no gaps in test data)
    try:
        assert len(daily) == 28, \
            f'daily resample of 28-day series should have 28 rows, got {len(daily)}'
        passed += 1; print(f'\u2705 Check 2: daily resample has {len(daily)} rows')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: weekly resample has fewer rows than daily
    try:
        weekly = resample_series(series, 'W')
        assert isinstance(weekly, pd.Series)
        assert len(weekly) < len(daily), \
            f'weekly ({len(weekly)}) should be shorter than daily ({len(daily)})'
        passed += 1; print(f'\u2705 Check 3: weekly has {len(weekly)} rows < {len(daily)} daily')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: multi_freq_summary defined, returns dict
    try:
        assert 'multi_freq_summary' in globals()
        summary = multi_freq_summary(series)
        assert isinstance(summary, dict), \
            f'expected dict, got {type(summary).__name__}'
        passed += 1; print('\u2705 Check 4: multi_freq_summary returns dict')
    except Exception as e:
        print(f'\u274c Check 4: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 5: dict has 'daily' and 'weekly' keys, both pd.Series
    try:
        for key in ('daily', 'weekly'):
            assert key in summary, f'missing key: {key!r}'
            assert isinstance(summary[key], pd.Series), \
                f'summary[{key!r}] should be Series'
        passed += 1; print("\u2705 Check 5: dict has 'daily' and 'weekly' Series")
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def resample_series(series: pd.Series, freq: str, agg: str = 'sum') -> pd.Series:
    return series.resample(freq).agg(agg)

def multi_freq_summary(series: pd.Series) -> dict:
    return {
        'daily':  resample_series(series, 'D'),
        'weekly': resample_series(series, 'W'),
    }
```

</details>